<a href="https://colab.research.google.com/github/esero88/eser-portfolio/blob/main/notebooks/global_ads_data_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Global Ads Performance: Data Preparation
**Goal:** Prepare the data for a campaign performance dashboard (Tableau).
**Source:** Kaggle, nudratabbas/global-ads-performance-google-meta-tiktok (synthetic data)
**Steps:** 1) Load 2) Overview 3) Quality checks 4) KPI validation 5) Export

## 1. Load

In [ ]:
import pandas as pd

URL = "https://raw.githubusercontent.com/esero88/eser-portfolio/main/data/raw/global_ads_performance_dataset_raw.csv"
df = pd.read_csv(URL)
df['date'] = pd.to_datetime(df['date'])

## 2. Overview

In [ ]:
print("Shape:", df.shape)
print("Date range:", df['date'].min().date(), "to", df['date'].max().date())
df.dtypes

Shape: (1800, 14)
Date range: 2024-01-01 to 2024-12-30


,0
date,datetime64[ns]
platform,object
campaign_type,object
industry,object
country,object
impressions,int64
clicks,int64
CTR,float64
CPC,float64
ad_spend,float64


In [ ]:
for c in ['platform', 'campaign_type', 'industry', 'country']:
    print(df[c].value_counts(), "\n")

platform
Google Ads    720
Meta Ads      630
TikTok Ads    450
Name: count, dtype: int64 

campaign_type
Search      477
Video       456
Shopping    447
Display     420
Name: count, dtype: int64 

industry
EdTech        372
SaaS          370
Fintech       361
E-commerce    349
Healthcare    348
Name: count, dtype: int64 

country
UK           266
USA          266
Canada       262
India        261
UAE          258
Germany      255
Australia    232
Name: count, dtype: int64 



## 3. Data quality checks

In [ ]:
checks = {
    "Missing values": df.isna().sum().sum(),
    "Duplicate rows": df.duplicated().sum(),
    "Clicks > impressions": (df['clicks'] > df['impressions']).sum(),
    "Conversions > clicks": (df['conversions'] > df['clicks']).sum(),
    "Ad spend <= 0": (df['ad_spend'] <= 0).sum(),
    "Zero conversions": (df['conversions'] == 0).sum(),
}
pd.Series(checks, name="count")

,count
Missing values,0
Duplicate rows,0
Clicks > impressions,0
Conversions > clicks,0
Ad spend <= 0,0
Zero conversions,0


In [ ]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
date,1800,2024-06-28 12:56:00,2024-01-01 00:00:00,2024-03-27 18:00:00,2024-06-26 00:00:00,2024-09-26 00:00:00,2024-12-30 00:00:00,NaN
impressions,1800.0,102919.018889,5059.0,54948.0,103653.0,150470.25,199650.0,55740.90069
clicks,1800.0,3962.675556,91.0,1678.0,3318.0,5628.0,16660.0,2941.858037
CTR,1800.0,0.038427,0.0089,0.0254,0.03555,0.0498,0.0956,0.017082
CPC,1800.0,1.572756,0.28,0.95,1.46,2.05,3.95,0.800872
ad_spend,1800.0,6171.527272,58.0,1966.5875,4393.86,8455.83,38453.32,5776.996958
conversions,1800.0,181.562222,2.0,59.0,130.0,252.25,1151.0,171.424239
CPA,1800.0,46.608961,4.8,20.2025,33.375,56.8125,335.86,41.185556
revenue,1800.0,30101.85045,142.69,7275.7575,18362.965,38963.385,295028.26,34560.032941
ROAS,1800.0,6.450367,0.13,2.17,4.295,8.2125,49.0,6.590986


## 4. KPI validation
Recalculate each KPI from raw columns and compare with the provided column.

In [ ]:
calc = pd.DataFrame({
    'CTR':  df['clicks'] / df['impressions'],
    'CPC':  df['ad_spend'] / df['clicks'],
    'CPA':  df['ad_spend'] / df['conversions'],
    'ROAS': df['revenue'] / df['ad_spend'],
})
for k in calc:
    diff = (df[k] - calc[k]).abs()
    print(f"{k}: max diff = {diff.max():.5f} | rows > 0.01: {(diff > 0.01).sum()}")

CTR: max diff = 0.00018 | rows > 0.01: 0
CPC: max diff = 0.00000 | rows > 0.01: 0
CPA: max diff = 0.00500 | rows > 0.01: 0
ROAS: max diff = 0.00499 | rows > 0.01: 0


**Result:** All provided KPI columns match the recalculated values (differences come from rounding only).

**Note for aggregation:** For any group (platform, country, month), compute KPIs from sums, not by averaging row-level ratios.
`ROAS = SUM(revenue) / SUM(ad_spend)`

## 5. Export for Tableau

In [ ]:
out = df.copy()
out['month'] = out['date'].dt.to_period('M').dt.to_timestamp()
out['quarter'] = out['date'].dt.to_period('Q').astype(str)
out.to_csv("global_ads_clean.csv", index=False)
out.head()

,date,platform,campaign_type,industry,country,impressions,clicks,CTR,CPC,ad_spend,conversions,CPA,revenue,ROAS,month,quarter
0,2024-01-21,Google Ads,Search,Fintech,UAE,59886,2113,0.0353,1.26,2662.38,159,16.74,4803.43,1.80,2024-01-01,2024Q1
1,2024-01-22,TikTok Ads,Search,EdTech,UK,135608,5220,0.0385,1.18,6159.60,411,14.99,64126.68,10.41,2024-01-01,2024Q1
2,2024-06-15,TikTok Ads,Video,Healthcare,USA,92313,5991,0.0649,0.85,5092.35,267,19.07,10489.07,2.06,2024-06-01,2024Q2
3,2024-01-02,TikTok Ads,Shopping,SaaS,Germany,83953,5935,0.0707,1.32,7834.20,296,26.47,50505.07,6.45,2024-01-01,2024Q1
4,2024-02-22,TikTok Ads,Search,Healthcare,UK,91807,4489,0.0489,1.93,8663.77,107,80.97,3369.53,0.39,2024-02-01,2024Q1


## Findings so far
- 1,800 rows, 14 columns, full year 2024, no missing values, no duplicates, no logic errors.
- Platform row counts are unbalanced (Google 720, Meta 630, TikTok 450), so compare platforms by rates, not totals.
- Sector and country distributions are very even, a typical sign of synthetic data.